<style>
.jp-RenderedHTMLCommon table, .dataframe {font-size: 0.95em;}
.jp-RenderedHTMLCommon h1 {border-bottom: 3px solid #1f5f99; padding-bottom: 6px;}
.jp-RenderedHTMLCommon h2 {border-bottom: 1px solid #9fb9d1; padding-bottom: 4px;}
.small-note {font-size: 0.92em; color: #444;}
</style>

# Chapter 1 Instructor Colab Notebook: Systems of Linear Equations
## Sections 1.1–1.3 · Theory, Visuals, Step-by-Step Computation, and Applications

**Course:** MATH 3260 — Linear Algebra  
**Primary platform:** Jupyter Notebook / JupyterLab / Google Colab  
**Audience:** Instructor version  

This notebook combines the mathematical content, classroom visuals, complete matrix calculations, Python demonstrations, interactive explorations, worksheets, and solutions for the opening chapter on linear systems.

> **Design rule used throughout:** Python supports the mathematics; it does not replace the mathematics. Every major algorithm is first developed by hand and then checked computationally.

> **How to use this notebook in class**
>
> Run the notebook from top to bottom before class. During class, use the Markdown cells as lecture notes, run the static plotting cells, and selectively use the interactive cells.
>
> Classroom questions are followed by **Reveal answer** buttons. Click a button only after students have had time to think or discuss. Student worksheets and complete instructor solutions are included at the end of each section.

## Notebook map

1. **Section 1.1:** Systems of Linear Equations  
2. **Section 1.2:** Gaussian Elimination and Gauss–Jordan Elimination  
3. **Section 1.3:** Applications of Linear Systems  
4. **Chapter summary and reusable Python reference**

Use Colab's table-of-contents panel to jump between headings.

In [ ]:
# Core setup: designed to run in Google Colab and standard Jupyter.
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Math, HTML

sp.init_printing()
np.set_printoptions(precision=4, suppress=True)

# Optional interactive support. Colab normally includes ipywidgets.
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Button, Output, VBox
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Interactive widgets are unavailable in this runtime. Static examples still work; reveal answers will display directly when called.")

# Enable third-party widgets when running in Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete.")

In [ ]:
# Reusable display, answer-reveal, and linear-algebra helper functions.
def show_augmented(M, title=None, split_col=None):
    """Display a SymPy augmented matrix with a vertical separator before split_col."""
    M = sp.Matrix(M)
    if title:
        display(Markdown(f"**{title}**"))
    if split_col is None:
        split_col = M.cols - 1
    if not (0 < split_col < M.cols):
        raise ValueError("split_col must be between 1 and the number of columns minus 1.")

    column_format = "c" * split_col + "|" + "c" * (M.cols - split_col)
    row_strings = []
    for i in range(M.rows):
        entries = [sp.latex(M[i, j]) for j in range(M.cols)]
        row_strings.append(" & ".join(entries))
    matrix_body = r" \\ ".join(row_strings)
    matrix_latex = (
        rf"\left[\begin{{array}}{{{column_format}}}"
        rf"{matrix_body}"
        rf"\end{{array}}\right]"
    )
    display(Math(matrix_latex))


def apply_row_operation(M, operation, description):
    """Apply an operation to a copy of M and display the result."""
    N = M.copy()
    operation(N)
    display(Markdown(f"**{description}**"))
    display(N)
    return N


def classify_system(A, b):
    """Classify Ax=b using exact ranks."""
    A = sp.Matrix(A)
    b = sp.Matrix(b)
    Aug = A.row_join(b)
    rank_A = A.rank()
    rank_aug = Aug.rank()
    n = A.cols
    if rank_A < rank_aug:
        status = "No solution (inconsistent)"
    elif rank_A == n:
        status = "Exactly one solution"
    else:
        status = "Infinitely many solutions"
    return {"rank(A)": rank_A, "rank([A|b])": rank_aug, "unknowns": n, "classification": status}


def plot_two_lines(a1, b1, c1, a2, b2, c2, title, xlim=(-5, 25), ylim=(-5, 25)):
    """Plot a1*x+b1*y=c1 and a2*x+b2*y=c2, including vertical-line cases."""
    x = np.linspace(xlim[0], xlim[1], 500)
    plt.figure(figsize=(7, 5))
    if abs(b1) > 1e-12:
        plt.plot(x, (c1-a1*x)/b1, label=f"{a1:g}x + {b1:g}y = {c1:g}")
    else:
        plt.axvline(c1/a1, label=f"{a1:g}x = {c1:g}")
    if abs(b2) > 1e-12:
        plt.plot(x, (c2-a2*x)/b2, label=f"{a2:g}x + {b2:g}y = {c2:g}")
    else:
        plt.axvline(c2/a2, label=f"{a2:g}x = {c2:g}")
    plt.axhline(0, linewidth=0.8)
    plt.axvline(0, linewidth=0.8)
    plt.xlim(*xlim); plt.ylim(*ylim)
    plt.xlabel("x"); plt.ylabel("y")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def exact_solve(A, b, symbols):
    """Solve Ax=b exactly with SymPy and return a dictionary."""
    equations = list(sp.Matrix(A) * sp.Matrix(symbols) - sp.Matrix(b))
    return sp.solve(equations, symbols, dict=True)


def reveal_answer(answer_markdown, button_text="Reveal answer"):
    """Create a click-to-reveal / click-to-hide answer button."""
    if not WIDGETS_AVAILABLE:
        # Dependency-free fallback: render a real HTML button rather than exposing
        # the answer immediately. Mistune is normally bundled with Jupyter.
        import uuid
        try:
            import mistune
            answer_html = mistune.html(answer_markdown)
        except Exception:
            import html
            answer_html = "<pre>" + html.escape(answer_markdown) + "</pre>"

        box_id = "answer_" + uuid.uuid4().hex
        btn_id = "button_" + uuid.uuid4().hex
        safe_label = button_text.replace('"', '&quot;')
        html_block = f"""
        <div style='margin:8px 0 14px 0;'>
          <button id='{btn_id}'
            style='padding:6px 12px;border:1px solid #4b87b9;border-radius:5px;background:#eef6ff;cursor:pointer;font-weight:600;'
            onclick="var box=document.getElementById('{box_id}'); var btn=document.getElementById('{btn_id}');
                     if(box.style.display==='none'){{box.style.display='block';btn.innerText='Hide answer';
                     if(window.MathJax && MathJax.typesetPromise){{MathJax.typesetPromise([box]);}}}}
                     else{{box.style.display='none';btn.innerText='{safe_label}';}}">
            {safe_label}
          </button>
          <div id='{box_id}' style='display:none;margin-top:10px;padding:10px 12px;border-left:4px solid #4b87b9;background:#fafcff;'>
            {answer_html}
          </div>
        </div>
        """
        display(HTML(html_block))
        return None

    button = widgets.ToggleButton(
        value=False,
        description=button_text,
        icon="eye",
        button_style="info",
        layout=widgets.Layout(width="180px")
    )
    out = widgets.Output()

    def _toggle(change):
        out.clear_output(wait=True)
        if change["new"]:
            button.description = "Hide answer"
            button.icon = "eye-slash"
            with out:
                display(Markdown(answer_markdown))
        else:
            button.description = button_text
            button.icon = "eye"

    button.observe(_toggle, names="value")
    display(widgets.VBox([button, out]))
    return button


def reveal_answer_key(key, button_text="Reveal answer"):
    """Reveal an answer stored in CH1_ANSWERS without exposing it in the classroom cell."""
    return reveal_answer(CH1_ANSWERS[key], button_text=button_text)

print("Helper functions loaded.")

In [ ]:
CH1_ANSWERS = {'quick-classification': '1. $4x-3y=21$: **linear**.  \n2. $x^2+y=9$: **nonlinear**, because a variable is squared.  \n3. $2x_1-x_2+x_3=20$: **linear**.  \n4. $xy-z=1$: **nonlinear**, because variables are multiplied.  \n5. $5u-2v+7w=0$: **linear** and **homogeneous**.  \n6. $\\sin x+y=3$: **nonlinear**, because a trigonometric function of a variable appears.', 'linear-means-line': 'No. The word **linear** describes the algebraic form of the equation, not the dimension of its graph.\n\n- In two variables, one linear equation graphs as a **line**.\n- In three variables, one linear equation graphs as a **plane**.\n- In higher dimensions, it graphs as a **hyperplane**.', 'geometry-cases': 'For two linear equations in two variables:\n\n- **One intersection:** the lines have different slopes.\n- **No solution:** the lines are distinct and parallel (same slope, different intercepts).\n- **Infinitely many solutions:** the two equations represent the same line (one equation is a nonzero scalar multiple of the other).', 'ref-check': 'For REF, check the staircase structure: nonzero rows above zero rows, each new leading entry to the right of the one above it, and zeros below each leading entry.\n\nTo reach RREF, each leading entry must additionally be $1$, and each pivot $1$ must be the only nonzero entry in its column. Therefore we may need to **scale pivot rows** and **eliminate entries above the pivots**.', '1.1-1': '(a) linear; (b) nonlinear because $xy$ multiplies variables; (c) linear and homogeneous; (d) nonlinear because a variable occurs in a denominator.', '1.1-2': 'Substitute $(x,y)=(4,5)$ into both equations:\n\n$$\n4+5=9,\n$$\n\nand\n\n$$\n2(4)-5=3.\n$$\n\nBoth equations are satisfied, so $(4,5)$ is a solution.', '1.1-3': 'Add the equations:\n\n$$\n(x+y)+(x-y)=20+2,\n$$\n\nso\n\n$$\n2x=22 \\quad\\Longrightarrow\\quad x=11.\n$$\n\nThen $11+y=20$, so $y=9$. The system has the unique solution\n\n$$\n(x,y)=(11,9).\n$$', '1.1-4': 'The first equation\n\n$$\n2x+4y=20\n$$\n\ndivides by $2$ to give\n\n$$\nx+2y=10.\n$$\n\nBut the second equation requires\n\n$$\nx+2y=12.\n$$\n\nThese constraints contradict each other, so the system has **no solution**.', '1.1-5': 'Let the free variables be\n\n$$\ny=s, \\qquad z=t.\n$$\n\nFrom\n\n$$\nx-y+2z=9,\n$$\n\nwe obtain\n\n$$\nx=9+s-2t.\n$$\n\nTherefore\n\n$$\n(x,y,z)=(9+s-2t,\\,s,\\,t), \\qquad s,t\\in\\mathbb R.\n$$', '1.1-6': 'Start with\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&30\\\\\n2&1&-1&19\\\\\n1&-1&2&12\n\\end{array}\n\\right].\n$$\n\nUse\n\n$$\nR_2\\leftarrow R_2-2R_1,\n\\qquad\nR_3\\leftarrow R_3-R_1,\n$$\n\nto obtain\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&30\\\\\n0&-1&-3&-41\\\\\n0&-2&1&-18\n\\end{array}\n\\right].\n$$\n\nThen\n\n$$\nR_3\\leftarrow R_3-2R_2\n$$\n\ngives\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&30\\\\\n0&-1&-3&-41\\\\\n0&0&7&64\n\\end{array}\n\\right].\n$$\n\nHence\n\n$$\nz=\\frac{64}{7},\\qquad\ny=\\frac{95}{7},\\qquad\nx=\\frac{51}{7}.\n$$', '1.1-7': 'Let $d$ be the drink price and $s$ the snack price. The model is\n\n$$\n2d+s=21,\n\\qquad\nd+2s=24.\n$$\n\nMultiply the second equation by $2$:\n\n$$\n2d+4s=48.\n$$\n\nSubtract the first equation:\n\n$$\n3s=27 \\quad\\Longrightarrow\\quad s=9.\n$$\n\nThen\n\n$$\n2d+9=21 \\quad\\Longrightarrow\\quad d=6.\n$$\n\nSo the modeled prices are **\\$6 per drink** and **\\$9 per snack**.', '1.2-1': 'The matrix has $2$ rows and $3$ columns, so its dimensions are\n\n$$\n2\\times 3.\n$$\n\nThe entry $a_{23}$ is in row $2$, column $3$, so\n\n$$\na_{23}=7.\n$$', '1.2-2': 'The augmented matrix is\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&-2&1&9\\\\\n3&1&-4&20\n\\end{array}\n\\right].\n$$', '1.2-3': 'Apply the row operation to **every entry** in Row 2:\n\n$$\n[3,7\\mid30]-3[1,2\\mid9]\n=[0,1\\mid3].\n$$\n\nTherefore\n\n$$\n\\left[\n\\begin{array}{cc|c}\n1&2&9\\\\\n0&1&3\n\\end{array}\n\\right].\n$$', '1.2-4': '- $A_1$ is **REF but not RREF**. It has the correct staircase pattern and zeros below pivots, but the leading entries in Rows 2 and 3 are not $1$, and pivot columns are not reduced above the pivots.\n- $A_2$ is **RREF**. Each nonzero row has a leading $1$, pivot positions move to the right, and each pivot is the only nonzero entry in its column.\n- $A_3$ is **neither**. The leading entry of Row 3 occurs in column $2$, which lies to the **left** of the leading entry of Row 2 in column $3$, violating the REF staircase condition.', '1.2-5': 'Begin with\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&21\\\\\n2&1&3&41\\\\\n1&3&2&38\n\\end{array}\n\\right].\n$$\n\nUse\n\n$$\nR_2\\leftarrow R_2-2R_1,\n\\qquad\nR_3\\leftarrow R_3-R_1,\n$$\n\nto obtain\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&21\\\\\n0&-1&1&-1\\\\\n0&2&1&17\n\\end{array}\n\\right].\n$$\n\nThen\n\n$$\nR_3\\leftarrow R_3+2R_2\n$$\n\ngives\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&1&1&21\\\\\n0&-1&1&-1\\\\\n0&0&3&15\n\\end{array}\n\\right].\n$$\n\nThus\n\n$$\nz=5,\n\\qquad\ny=6,\n\\qquad\nx=10.\n$$', '1.2-6': 'The RREF is\n\n$$\n\\left[\n\\begin{array}{ccc|c}\n1&0&0&9\\\\\n0&1&0&11\\\\\n0&0&1&11\n\\end{array}\n\\right].\n$$\n\nTherefore\n\n$$\n(x,y,z)=(9,11,11).\n$$', '1.2-7': 'The second equation is exactly $2$ times the first equation, so it adds no new constraint. The system has infinitely many solutions.\n\nLet\n\n$$\ny=s, \\qquad z=t.\n$$\n\nThen\n\n$$\nx=9-2s+t.\n$$\n\nTherefore\n\n$$\n(x,y,z)=(9-2s+t,\\,s,\\,t), \\qquad s,t\\in\\mathbb R.\n$$', '1.2-8': 'The second equation is twice the first, so the only independent equation is\n\n$$\nx+y+z=0.\n$$\n\nLet\n\n$$\ny=s, \\qquad z=t.\n$$\n\nThen\n\n$$\nx=-s-t.\n$$\n\nHence\n\n$$\n(x,y,z)=(-s-t,\\,s,\\,t), \\qquad s,t\\in\\mathbb R.\n$$\n\nThe trivial solution occurs at $s=t=0$; all other parameter choices give nontrivial solutions.', '1.3-1': 'Let\n\n$$\ny=mx+b.\n$$\n\nUsing $(2,21)$ and $(5,9)$ gives\n\n$$\n2m+b=21,\n\\qquad\n5m+b=9.\n$$\n\nSubtracting yields\n\n$$\n3m=-12 \\quad\\Longrightarrow\\quad m=-4.\n$$\n\nThen\n\n$$\nb=29.\n$$\n\nTherefore\n\n$$\ny=-4x+29.\n$$', '1.3-2': 'Substituting the three points into\n\n$$\np(x)=ax^2+bx+c\n$$\n\ngives\n\n$$\nc=1,\n\\qquad\na+b+c=9,\n\\qquad\n4a+2b+c=21.\n$$\n\nSince $c=1$,\n\n$$\na+b=8,\n\\qquad\n2a+b=10.\n$$\n\nSubtracting gives $a=2$, and then $b=6$. Thus\n\n$$\np(x)=2x^2+6x+1.\n$$', '1.3-3': 'Conservation requires total inflow to equal total outflow. Therefore\n\n$$\nx_1+x_2=50.\n$$', '1.3-4': 'Let\n\n$$\nx_1=t.\n$$\n\nThen\n\n$$\nx_2=50-t\n$$\n\nfrom the first equation, and\n\n$$\nx_3=t+9\n$$\n\nfrom the second equation. The third equation is automatically satisfied because\n\n$$\n(50-t)+(t+9)=59.\n$$\n\nHence the one-parameter flow family is\n\n$$\n(x_1,x_2,x_3)=(t,\\,50-t,\\,t+9).\n$$\n\nThe parameter represents flexibility in the internal distribution of flow.', '1.3-5': 'For all flows to be nonnegative,\n\n$$\nt\\ge 0,\n\\qquad\n50-t\\ge0,\n\\qquad\nt+9\\ge0.\n$$\n\nThe first two inequalities give\n\n$$\n0\\le t\\le50,\n$$\n\nand the third is automatically satisfied on this interval. Therefore the feasible range is\n\n$$\n\\boxed{0\\le t\\le50}.\n$$', '1.3-6': 'From\n\n$$\nI_1+I_2=21,\n$$\n\nwe have\n\n$$\nI_1=21-I_2.\n$$\n\nThe junction equation\n\n$$\nI_1-I_2+I_3=0\n$$\n\ngives\n\n$$\nI_3=2I_2-21.\n$$\n\nSubstitute into\n\n$$\nI_2+2I_3=20:\n$$\n\n$$\nI_2+2(2I_2-21)=20,\n$$\n\nso\n\n$$\n5I_2=62,\n\\qquad\nI_2=\\frac{62}{5}.\n$$\n\nThen\n\n$$\nI_1=\\frac{43}{5},\n\\qquad\nI_3=\\frac{19}{5}.\n$$', '1.3-7': 'Subtract the first equation from the second:\n\n$$\nr=9.\n$$\n\nThen\n\n$$\ng+b=12.\n$$\n\nThe third equation becomes\n\n$$\n9+2g+3b=44,\n$$\n\nso\n\n$$\n2g+3b=35.\n$$\n\nSubtract\n\n$$\n2(g+b)=24\n$$\n\nto obtain\n\n$$\nb=11.\n$$\n\nHence $g=1$, and\n\n$$\n(r,g,b)=(9,1,11).\n$$\n\nVerification:\n\n$$\n9+1+11=21,\n$$\n\n$$\n2(9)+1+11=30,\n$$\n\nand\n\n$$\n9+2(1)+3(11)=44.\n$$', 'application-discussion': "The linearity comes from how the unknown quantities appear:\n\n- **Curve fitting:** the unknown coefficients $a,b,c,\\ldots$ appear only to the first power and are not multiplied together.\n- **Network flow:** conservation equations use sums and differences of the unknown flows.\n- **Circuits with fixed resistances:** Ohm's law $V=RI$ is linear in current when $R$ is fixed, and Kirchhoff's laws combine currents and voltage drops using linear sums.\n\nIf coefficients themselves depended nonlinearly on the unknowns, the model would no longer be a linear system."}
print(f"Loaded {len(CH1_ANSWERS)} instructor answer keys.")

> **Notation used in code**
>
> `A` denotes a coefficient matrix, `b` denotes the right-hand-side vector, and `Aug` denotes an augmented matrix. SymPy is used when exact fractions and symbolic parameters matter. NumPy is used for numerical computation and plotting.

# Section 1.1: Systems of Linear Equations

## Learning objectives

By the end of this section, students should be able to:

- recognize linear and nonlinear equations;
- explain what it means to solve a system;
- distinguish one solution, no solution, and infinitely many solutions;
- connect algebraic solutions with line and plane intersections;
- write parametric solution sets;
- classify systems as consistent or inconsistent;
- perform a first elimination calculation by hand;
- verify solutions with Python.

> **Why systems matter**
>
> A system is a collection of constraints that must hold simultaneously. The same structure appears in pricing, mixtures, traffic, circuits, computer graphics, data fitting, and resource allocation.

## 1. Linear equations

A linear equation in variables $x_1,x_2,\ldots,x_n$ has the form

$$
a_1x_1+a_2x_2+\cdots+a_nx_n=b,
$$

where the coefficients $a_1,\ldots,a_n$ and the constant $b$ are fixed numbers. Variables occur only to the first power; variables are not multiplied together, placed in denominators, or used inside nonlinear functions.

> **Definition: System and Solution**
>
> A **system of linear equations** is a collection of linear equations involving the same variables.
>
> A **solution** is an ordered list of values that satisfies every equation in the system at the same time.

### Quick classification: linear or nonlinear?

Decide before running the cell.

1. $4x-3y=21$
2. $x^2+y=9$
3. $2x_1-x_2+x_3=20$
4. $xy-z=1$
5. $5u-2v+7w=0$
6. $\sin x+y=3$

In [ ]:
reveal_answer_key("quick-classification", "Reveal classification")

### Classroom Question

**Does linear always mean a line?**

Give students a moment to discuss before revealing the answer.

In [ ]:
reveal_answer_key("linear-means-line")

## 2. Worked example: one solution

Solve

$$
\begin{cases}
u+v=21,\\
u-v=-3.
\end{cases}
$$

Add the equations:

$$
(u+v)+(u-v)=21+(-3),
$$

so

$$
2u=18 \quad\Longrightarrow\quad u=9.
$$

Substitute $u=9$ into $u+v=21$:

$$
9+v=21 \quad\Longrightarrow\quad v=12.
$$

Therefore,

$$
(u,v)=(9,12).
$$

The numbers are Spurs-inspired, but the system itself is an original teaching example.

In [ ]:
u, v = sp.symbols('u v')
solution_11 = sp.solve([sp.Eq(u+v, 21), sp.Eq(u-v, -3)], [u, v], dict=True)
solution_11

In [ ]:
plot_two_lines(1, 1, 21, 1, -1, -3,
               title="One solution: the lines intersect once",
               xlim=(0, 22), ylim=(0, 22))

### Verify rather than merely trust

A candidate solution must satisfy **both** equations.

In [ ]:
candidate = {u: 9, v: 12}
checks = [
    sp.simplify((u+v-21).subs(candidate)),
    sp.simplify((u-v+3).subs(candidate)),
]
print("Residuals:", checks)
print("Both residuals are zero, so the candidate solves the system.")

## 3. Interactive geometry of two equations

Use the sliders to change the slopes and intercepts. Ask students to create:

- one intersection;
- parallel distinct lines;
- the same line twice.

In [ ]:
if WIDGETS_AVAILABLE:
    @interact(
        m1=FloatSlider(value=1.0, min=-3, max=3, step=0.25, description='m1'),
        b1=FloatSlider(value=2.0, min=-10, max=10, step=0.5, description='b1'),
        m2=FloatSlider(value=-1.0, min=-3, max=3, step=0.25, description='m2'),
        b2=FloatSlider(value=8.0, min=-10, max=10, step=0.5, description='b2')
    )
    def explore_lines(m1, b1, m2, b2):
        x = np.linspace(-10, 10, 500)
        plt.figure(figsize=(7, 5))
        plt.plot(x, m1*x+b1, label=f"y={m1:.2f}x+{b1:.2f}")
        plt.plot(x, m2*x+b2, label=f"y={m2:.2f}x+{b2:.2f}")
        plt.axhline(0, linewidth=0.8); plt.axvline(0, linewidth=0.8)
        plt.xlim(-10,10); plt.ylim(-10,10); plt.grid(alpha=0.3); plt.legend()
        if abs(m1-m2) < 1e-10:
            status = "same line" if abs(b1-b2) < 1e-10 else "parallel: no solution"
        else:
            xi = (b2-b1)/(m1-m2); yi=m1*xi+b1
            status = f"one solution near ({xi:.2f}, {yi:.2f})"
            plt.scatter([xi],[yi], s=70)
        plt.title(status)
        plt.show()
else:
    print("Open this notebook in Colab to use the line sliders.")

### Classroom Check

After students experiment with the sliders, ask: **What algebraic conditions produce each of the three geometric cases?**

In [ ]:
reveal_answer_key("geometry-cases", "Reveal conditions")

## 4. The three possible outcomes

For a linear system in two variables, exactly one of the following occurs:

| Geometry | Algebraic meaning | Number of solutions |
|---|---|---:|
| Distinct intersecting lines | Two independent constraints | One |
| Distinct parallel lines | Contradictory constraints | None |
| Same line | One equation repeats the other | Infinitely many |

In [ ]:
x = np.linspace(-2, 12, 400)

plt.figure(figsize=(7,4.5))
plt.plot(x, 10-x, label='x+y=10')
plt.plot(x, x+2, label='y=x+2')
plt.scatter([4],[6],s=70)
plt.axhline(0,linewidth=0.8); plt.axvline(0,linewidth=0.8)
plt.xlim(-2,12); plt.ylim(-2,14); plt.grid(alpha=0.3)
plt.xlabel('x'); plt.ylabel('y'); plt.title('One solution'); plt.legend(); plt.show()

plt.figure(figsize=(7,4.5))
plt.plot(x, 10-2*x, label='2x+y=10')
plt.plot(x, 12-2*x, label='2x+y=12')
plt.axhline(0,linewidth=0.8); plt.axvline(0,linewidth=0.8)
plt.xlim(-2,12); plt.ylim(-2,14); plt.grid(alpha=0.3)
plt.xlabel('x'); plt.ylabel('y'); plt.title('No solution'); plt.legend(); plt.show()

plt.figure(figsize=(7,4.5))
plt.plot(x, 9+0.5*x, linewidth=4, alpha=0.55, label='same line')
plt.plot(x, 9+0.5*x, linestyle='--')
plt.axhline(0,linewidth=0.8); plt.axvline(0,linewidth=0.8)
plt.xlim(-2,12); plt.ylim(-2,16); plt.grid(alpha=0.3)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Infinitely many solutions'); plt.legend(); plt.show()

### Worked example: no solution

$$
\begin{cases}
2x+y=20,\\
4x+2y=42.
\end{cases}
$$

Multiplying the first equation by $2$ gives

$$
4x+2y=40,
$$

but the second equation requires $4x+2y=42$. The same expression cannot equal both $40$ and $42$. Therefore the system is inconsistent.

In [ ]:
print(classify_system([[2,1],[4,2]], [20,42]))
plot_two_lines(2,1,20,4,2,42,"No solution: parallel constraints", xlim=(-2,12), ylim=(-2,24))

### Worked example: infinitely many solutions

$$
\begin{cases}
3x-y=9,\\
6x-2y=18.
\end{cases}
$$

The second equation is twice the first, so both equations describe the same line. Let $x=t$. Then

$$
3t-y=9 \quad\Longrightarrow\quad y=3t-9.
$$

The complete solution set is

$$
(x,y)=(t,3t-9),\qquad t\in\mathbb R.
$$

In [ ]:
x, y, t = sp.symbols('x y t')
solution_inf = sp.linsolve([3*x-y-9, 6*x-2*y-18], (x,y))
solution_inf

> **Parameter Interpretation**
>
> A parameter does not mean “unknown forever.” It means the system allows a family of solutions. Choosing any real value of the parameter produces one member of that family.

## 5. One equation in three variables

Consider

$$
x+2y-z=21.
$$

There are three variables but only one constraint. Let

$$
y=s,\qquad z=t.
$$

Then

$$
x=21-2s+t.
$$

Therefore

$$
(x,y,z)=(21-2s+t,s,t),\qquad s,t\in\mathbb R.
$$

Geometrically, the solution set is a plane in $\mathbb R^3$.

In [ ]:
# Plot a portion of x + 2y - z = 21.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
s_vals = np.linspace(-2, 12, 25)
t_vals = np.linspace(-2, 12, 25)
S, T = np.meshgrid(s_vals, t_vals)
X = 21 - 2*S + T

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, S, T, alpha=0.65)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(r"Solution plane: $x+2y-z=21$")
plt.show()

In [ ]:
# Generate several exact points on the plane using parameter values.
parameter_choices = [(-1,1), (0,0), (3,2), (9,1), (20,21)]
points = []
for s_value, t_value in parameter_choices:
    x_value = 21 - 2*s_value + t_value
    points.append((x_value, s_value, t_value))
points

## 6. Application: Spurs fan-shop bundles

Let

- $c$ = price of a cap;
- $p$ = price of a poster.

Suppose two original promotional bundles are priced as follows:

$$
\begin{cases}
2c+p=30,\\
c+2p=33.
\end{cases}
$$

The numbers are chosen so the solution uses familiar Spurs-related values, but this is not actual store pricing.

Solve by elimination:

Multiply the second equation by $2$:

$$
2c+4p=66.
$$

Subtract the first equation:

$$
(2c+4p)-(2c+p)=66-30,
$$

so

$$
3p=36\quad\Longrightarrow\quad p=12.
$$

Then

$$
2c+12=30\quad\Longrightarrow\quad c=9.
$$

Thus the modeled prices are $c=9$ dollars and $p=12$ dollars.

In [ ]:
A_shop = sp.Matrix([[2,1],[1,2]])
b_shop = sp.Matrix([30,33])
shop_solution = A_shop.LUsolve(b_shop)
shop_solution

## 7. Detailed preview: elimination in three variables

Solve

$$
\begin{cases}
x+y+z=31,\\
2x-y+3z=36,\\
-x+2y+z=-2.
\end{cases}
$$

Write the augmented matrix:

$$
\left[
\begin{array}{ccc|c}
1&1&1&31\\
2&-1&3&36\\
-1&2&1&-2
\end{array}
\right].
$$

In [ ]:
M11_0 = sp.Matrix([[1,1,1,31],[2,-1,3,36],[-1,2,1,-2]])
show_augmented(M11_0, "Initial augmented matrix")

Eliminate $x$ from Row 2 using

$$
R_2\leftarrow R_2-2R_1.
$$

Entry-by-entry:

$$
[2,-1,3\mid36]-2[1,1,1\mid31]=[0,-3,1\mid-26].
$$

In [ ]:
M11_1 = M11_0.copy()
M11_1[1,:] = M11_1[1,:] - 2*M11_1[0,:]
show_augmented(M11_1, r"After $R_2 \leftarrow R_2-2R_1$")

Eliminate $x$ from Row 3 using

$$
R_3\leftarrow R_3+R_1.
$$

Entry-by-entry:

$$
[-1,2,1\mid-2]+[1,1,1\mid31]=[0,3,2\mid29].
$$

In [ ]:
M11_2 = M11_1.copy()
M11_2[2,:] = M11_2[2,:] + M11_2[0,:]
show_augmented(M11_2, r"After $R_3 \leftarrow R_3+R_1$")

Eliminate $y$ from Row 3 using

$$
R_3\leftarrow R_3+R_2.
$$

$$
[0,3,2\mid29]+[0,-3,1\mid-26]=[0,0,3\mid3].
$$

In [ ]:
M11_3 = M11_2.copy()
M11_3[2,:] = M11_3[2,:] + M11_3[1,:]
show_augmented(M11_3, r"After $R_3 \leftarrow R_3+R_2$")

Now back-substitute:

$$
3z=3\quad\Longrightarrow\quad z=1.
$$

From Row 2,

$$
-3y+z=-26
$$

so

$$
-3y+1=-26\quad\Longrightarrow\quad -3y=-27\quad\Longrightarrow\quad y=9.
$$

From Row 1,

$$
x+y+z=31,
$$

so

$$
x+9+1=31\quad\Longrightarrow\quad x=21.
$$

Therefore,

$$
(x,y,z)=(21,9,1).
$$

In [ ]:
A11 = sp.Matrix([[1,1,1],[2,-1,3],[-1,2,1]])
b11 = sp.Matrix([31,36,-2])
print("Exact solution:")
display(A11.LUsolve(b11))
print("Verification A*x:")
display(A11*A11.LUsolve(b11))

In [ ]:
# Interactive step viewer for the preceding elimination.
steps_11 = [M11_0, M11_1, M11_2, M11_3]
descriptions_11 = [
    "Initial augmented matrix",
    r"Eliminate x from Row 2: R2 <- R2 - 2R1",
    r"Eliminate x from Row 3: R3 <- R3 + R1",
    r"Eliminate y from Row 3: R3 <- R3 + R2",
]
if WIDGETS_AVAILABLE:
    @interact(step=IntSlider(value=0, min=0, max=3, step=1, description='step'))
    def view_section11_step(step):
        display(Markdown(f"**{descriptions_11[step]}**"))
        display(steps_11[step])
else:
    print("Enable ipywidgets to use the step slider.")

> **Common Mistakes in Section 1.1**
>
> - Checking only one equation.
> - Assuming every system has one solution.
> - Giving one point when a parameterized family is required.
> - Confusing a free variable with a pivot variable.
> - Reporting a numerical answer without interpreting its meaning in context.

## Section 1.1 terminology

| Term | Meaning |
|---|---|
| Linear equation | An equation of the form $a_1x_1+\cdots+a_nx_n=b$ |
| System of linear equations | Several linear equations involving the same variables |
| Solution | Values satisfying every equation simultaneously |
| Solution set | The collection of all solutions |
| Consistent system | A system with at least one solution |
| Inconsistent system | A system with no solution |
| Unique solution | Exactly one solution |
| Free variable | A variable allowed to take arbitrary parameter values |
| Parameter | A symbol representing freely chosen values in a solution family |
| Parametric form | A description of all solutions using one or more parameters |
| Plane | The solution set of one nondegenerate linear equation in three variables |

## Section 1.1 worksheet

1. Decide whether each equation is linear:  
   (a) $3x-4y+z=21$; (b) $xy+z=9$; (c) $x_1-2x_2=0$; (d) $1/x+y=4$.
2. Check whether $(4,5)$ solves $x+y=9$ and $2x-y=3$.
3. Classify and solve: $x+y=20$, $x-y=2$.
4. Classify: $2x+4y=20$, $x+2y=12$.
5. Write the solution of $x-y+2z=9$ in parametric form.
6. Solve by elimination:
   $$
   \begin{cases}
   x+y+z=30,\\
   2x+y-z=19,\\
   x-y+2z=12.
   \end{cases}
   $$
7. A fictional arena bundle contains two drinks and one snack for \$21, while one drink and two snacks cost \$24. Find the modeled item prices.

## Section 1.1 Instructor Answer Reveals

Reveal one solution at a time during class.

In [ ]:
reveal_answer_key("1.1-1", "Reveal Problem 1")
reveal_answer_key("1.1-2", "Reveal Problem 2")
reveal_answer_key("1.1-3", "Reveal Problem 3")
reveal_answer_key("1.1-4", "Reveal Problem 4")
reveal_answer_key("1.1-5", "Reveal Problem 5")
reveal_answer_key("1.1-6", "Reveal Problem 6")
reveal_answer_key("1.1-7", "Reveal Problem 7")

# Section 1.2: Gaussian Elimination and Gauss–Jordan Elimination

## Learning objectives

Students should be able to:

- identify matrix dimensions and entries;
- construct coefficient and augmented matrices;
- perform all three elementary row operations;
- distinguish row-echelon form from reduced row-echelon form;
- solve systems using Gaussian elimination and back-substitution;
- solve systems using Gauss–Jordan elimination;
- detect inconsistent and dependent systems from row-reduced matrices;
- analyze homogeneous systems;
- reproduce each matrix step manually and verify it with Python.

## 1. Matrices as organized data

An $m\times n$ matrix has $m$ rows and $n$ columns:

$$
A=\begin{bmatrix}
a_{11}&a_{12}&\cdots&a_{1n}\\
a_{21}&a_{22}&\cdots&a_{2n}\\
\vdots&\vdots&\ddots&\vdots\\
a_{m1}&a_{m2}&\cdots&a_{mn}
\end{bmatrix}.
$$

The entry $a_{ij}$ lies in row $i$ and column $j$.

In [ ]:
A_demo = sp.Matrix([[21,9,1],[20,12,6]])
print("Matrix:")
display(A_demo)
print("Rows, columns =", A_demo.shape)
print("Entry in row 2, column 3 =", A_demo[1,2])

> **Coefficient and Augmented Matrices**
>
> For
>
> $$
> A\mathbf{x}=\mathbf{b},
> $$
>
> the **coefficient matrix** contains only the variable coefficients. The **augmented matrix**
>
> $$
> [A\mid\mathbf{b}]
> $$
>
> appends the constants as a final column.

For example,

$$
\begin{cases}
x+2y-z=21,\\
3x-y+4z=20
\end{cases}
$$

has coefficient matrix

$$
A=\begin{bmatrix}1&2&-1\\3&-1&4\end{bmatrix}
$$

and augmented matrix

$$
[A\mid b]=\left[\begin{array}{ccc|c}1&2&-1&21\\3&-1&4&20\end{array}\right].
$$

In [ ]:
A = sp.Matrix([[1,2,-1],[3,-1,4]])
b = sp.Matrix([21,20])
Aug = A.row_join(b)
print("Coefficient matrix A:"); display(A)
print("Right-hand side b:"); display(b)
print("Augmented matrix [A|b]:"); display(Aug)

## 2. Elementary row operations

Each legal row operation preserves the solution set.

1. **Row replacement:** $R_i\leftarrow R_i+kR_j$.
2. **Row interchange:** $R_i\leftrightarrow R_j$.
3. **Row scaling:** $R_i\leftarrow kR_i$, where $k\ne0$.

In [ ]:
M = sp.Matrix([[1,2,21],[3,-1,20]])
show_augmented(M, "Starting matrix")

M_replace = M.copy()
M_replace[1,:] = M_replace[1,:] - 3*M_replace[0,:]
show_augmented(M_replace, r"Row replacement: R2 <- R2 - 3R1")

M_swap = M_replace.copy()
M_swap.row_swap(0,1)
show_augmented(M_swap, r"Row interchange: R1 <-> R2")

M_scale = M_swap.copy()
M_scale[0,:] = sp.Rational(-1,7)*M_scale[0,:]
show_augmented(M_scale, r"Row scaling: R1 <- (-1/7)R1")

> **Why Row Operations Are Valid**
>
> Replacing one equation by a nonzero multiple, swapping equation order, or adding a multiple of one equation to another does not change which variable values satisfy the entire system.

## 3. Row-echelon form and reduced row-echelon form

A matrix is in **row-echelon form (REF)** when:

1. all nonzero rows are above zero rows;
2. each leading entry is to the right of the leading entry above it;
3. all entries below a leading entry are zero.

It is in **reduced row-echelon form (RREF)** when, additionally:

4. each leading entry is $1$;
5. each pivot $1$ is the only nonzero entry in its column.

In [ ]:
REF_example = sp.Matrix([[1,2,-1,4],[0,3,5,7],[0,0,2,6]])
RREF_example = sp.Matrix([[1,0,0,9],[0,1,0,20],[0,0,1,21]])
not_REF = sp.Matrix([[1,2,0],[0,0,1],[0,3,0]])

print("REF but not RREF:"); display(REF_example)
print("RREF:"); display(RREF_example)
print("Not REF:"); display(not_REF)

### Classroom Check: REF versus RREF

Point to each pivot in the displayed matrices and ask students:

1. Does the first matrix satisfy every REF condition?
2. What additional work is needed to turn it into RREF?

Reveal the explanation after students respond.

In [ ]:
reveal_answer_key("ref-check", "Reveal REF/RREF explanation")

## 4. Gaussian elimination: full application example

A fictional Spurs fan shop sells three items:

- $c$ = cap price;
- $s$ = shirt price;
- $p$ = poster price.

Three original bundles produce the system

$$
\begin{cases}
c+s+p=50,\\
2c+s+3p=101,\\
c+2s+4p=133.
\end{cases}
$$

The goal is not the realism of the bundle design; the goal is to model three unknown quantities and execute every elimination step.

### Step 0: write the augmented matrix

$$
\left[
\begin{array}{ccc|c}
1&1&1&50\\
2&1&3&101\\
1&2&4&133
\end{array}
\right].
$$

In [ ]:
M12_0 = sp.Matrix([[1,1,1,50],[2,1,3,101],[1,2,4,133]])
show_augmented(M12_0, "Step 0")

### Step 1: eliminate the first entry of Row 2

Use

$$
R_2\leftarrow R_2-2R_1.
$$

Compute every entry:

$$
[2,1,3\mid101]-2[1,1,1\mid50]
=[0,-1,1\mid1].
$$

In [ ]:
M12_1 = M12_0.copy()
M12_1[1,:] = M12_1[1,:] - 2*M12_1[0,:]
show_augmented(M12_1, r"Step 1: R2 <- R2 - 2R1")

### Step 2: eliminate the first entry of Row 3

Use

$$
R_3\leftarrow R_3-R_1.
$$

$$
[1,2,4\mid133]-[1,1,1\mid50]
=[0,1,3\mid83].
$$

In [ ]:
M12_2 = M12_1.copy()
M12_2[2,:] = M12_2[2,:] - M12_2[0,:]
show_augmented(M12_2, r"Step 2: R3 <- R3 - R1")

### Step 3: eliminate the second entry of Row 3

Row 2 has $-1$ in the pivot position and Row 3 has $1$. Add Row 2 to Row 3:

$$
R_3\leftarrow R_3+R_2.
$$

$$
[0,1,3\mid83]+[0,-1,1\mid1]
=[0,0,4\mid84].
$$

In [ ]:
M12_3 = M12_2.copy()
M12_3[2,:] = M12_3[2,:] + M12_3[1,:]
show_augmented(M12_3, r"Step 3: R3 <- R3 + R2")

The matrix is now in row-echelon form. Translate each row back into an equation:

$$
\begin{aligned}
c+s+p&=50,\\
-s+p&=1,\\
4p&=84.
\end{aligned}
$$

### Back-substitution

From Row 3:

$$
p=\frac{84}{4}=21.
$$

From Row 2:

$$
-s+p=1
$$

so

$$
-s+21=1\quad\Longrightarrow\quad -s=-20\quad\Longrightarrow\quad s=20.
$$

From Row 1:

$$
c+20+21=50\quad\Longrightarrow\quad c=9.
$$

Thus

$$
(c,s,p)=(9,20,21).
$$

In [ ]:
A_bundle = sp.Matrix([[1,1,1],[2,1,3],[1,2,4]])
b_bundle = sp.Matrix([50,101,133])
solution_bundle = A_bundle.LUsolve(b_bundle)
print("SymPy verification:"); display(solution_bundle)
print("A*x ="); display(A_bundle*solution_bundle)

In [ ]:
steps_12 = [M12_0, M12_1, M12_2, M12_3]
descriptions_12 = [
    "Step 0: original augmented matrix",
    "Step 1: R2 <- R2 - 2R1",
    "Step 2: R3 <- R3 - R1",
    "Step 3: R3 <- R3 + R2 (REF reached)",
]
if WIDGETS_AVAILABLE:
    @interact(step=IntSlider(value=0, min=0, max=3, description='step'))
    def gaussian_step_viewer(step):
        display(Markdown(f"**{descriptions_12[step]}**"))
        display(steps_12[step])
else:
    print("Enable ipywidgets to use the Gaussian-elimination step slider.")

## 5. Gauss–Jordan elimination: continue all the way to RREF

Solve

$$
\begin{cases}
x+y+z=50,\\
x+2y+z=71,\\
2x+y+3z=88.
\end{cases}
$$

We will reduce the augmented matrix until every pivot column has zeros above and below its pivot.

In [ ]:
G0 = sp.Matrix([[1,1,1,50],[1,2,1,71],[2,1,3,88]])
show_augmented(G0, "Initial matrix")

### Forward elimination

First,

$$
R_2\leftarrow R_2-R_1,
$$

which gives

$$
[1,2,1\mid71]-[1,1,1\mid50]=[0,1,0\mid21].
$$

In [ ]:
G1 = G0.copy(); G1[1,:] = G1[1,:] - G1[0,:]
show_augmented(G1, r"After R2 <- R2 - R1")

Next,

$$
R_3\leftarrow R_3-2R_1,
$$

so

$$
[2,1,3\mid88]-2[1,1,1\mid50]=[0,-1,1\mid-12].
$$

In [ ]:
G2 = G1.copy(); G2[2,:] = G2[2,:] - 2*G2[0,:]
show_augmented(G2, r"After R3 <- R3 - 2R1")

Now eliminate the $-1$ below the second pivot:

$$
R_3\leftarrow R_3+R_2,
$$

so

$$
[0,-1,1\mid-12]+[0,1,0\mid21]=[0,0,1\mid9].
$$

In [ ]:
G3 = G2.copy(); G3[2,:] = G3[2,:] + G3[1,:]
show_augmented(G3, r"After R3 <- R3 + R2 (REF)")

### Backward elimination to RREF

Eliminate $z$ from Row 1:

$$
R_1\leftarrow R_1-R_3.
$$

In [ ]:
G4 = G3.copy(); G4[0,:] = G4[0,:] - G4[2,:]
show_augmented(G4, r"After R1 <- R1 - R3")

Eliminate $y$ from Row 1:

$$
R_1\leftarrow R_1-R_2.
$$

In [ ]:
G5 = G4.copy(); G5[0,:] = G5[0,:] - G5[1,:]
show_augmented(G5, r"After R1 <- R1 - R2 (RREF)")

The RREF is

$$
\left[
\begin{array}{ccc|c}
1&0&0&20\\
0&1&0&21\\
0&0&1&9
\end{array}
\right],
$$

so

$$
(x,y,z)=(20,21,9).
$$

In [ ]:
print("SymPy rref result and pivot columns:")
G0.rref()

## 6. Gaussian versus Gauss–Jordan

| Method | Stopping point | Final work |
|---|---|---|
| Gaussian elimination | Row-echelon form | Back-substitution |
| Gauss–Jordan elimination | Reduced row-echelon form | Read solution directly |

Gaussian elimination often uses fewer operations. Gauss–Jordan produces a more informative final matrix.

## 7. Detecting an inconsistent system

Consider

$$
\begin{cases}
x+2y=21,\\
2x+4y=43.
\end{cases}
$$

Start with

$$
\left[\begin{array}{cc|c}1&2&21\\2&4&43\end{array}\right].
$$

Use

$$
R_2\leftarrow R_2-2R_1.
$$

Then

$$
[2,4\mid43]-2[1,2\mid21]=[0,0\mid1].
$$

The last row means $0=1$, which is impossible.

In [ ]:
I0 = sp.Matrix([[1,2,21],[2,4,43]])
I1 = I0.copy(); I1[1,:] = I1[1,:] - 2*I1[0,:]
show_augmented(I0, "Before elimination")
show_augmented(I1, "Contradictory row appears")
print(classify_system([[1,2],[2,4]],[21,43]))

## 8. Infinitely many solutions from RREF

Consider

$$
\begin{cases}
x+y+z=21,\\
2x+2y+2z=42,\\
x-y+z=1.
\end{cases}
$$

The augmented matrix is

$$
\left[\begin{array}{ccc|c}1&1&1&21\\2&2&2&42\\1&-1&1&1\end{array}\right].
$$

In [ ]:
D0 = sp.Matrix([[1,1,1,21],[2,2,2,42],[1,-1,1,1]])
D1 = D0.copy(); D1[1,:] = D1[1,:]-2*D1[0,:]; D1[2,:]=D1[2,:]-D1[0,:]
D2 = D1.copy(); D2[2,:] = sp.Rational(-1,2)*D2[2,:]
D3 = D2.copy(); D3[0,:] = D3[0,:]-D3[2,:]
show_augmented(D0, "Initial")
show_augmented(D1, "After eliminating x from Rows 2 and 3")
show_augmented(D2, "Scale Row 3")
show_augmented(D3, "RREF after clearing y above its pivot")

The RREF equations are

$$
x+z=11,\qquad y=10.
$$

Let $z=t$. Then

$$
x=11-t,
$$

so

$$
(x,y,z)=(11-t,10,t),\qquad t\in\mathbb R.
$$

In [ ]:
x,y,z = sp.symbols('x y z')
sp.linsolve((sp.Matrix([[1,1,1],[2,2,2],[1,-1,1]]), sp.Matrix([21,42,1])), (x,y,z))

## 9. Homogeneous systems

A homogeneous system has the form

$$
A\mathbf{x}=\mathbf{0}.
$$

It always has the **trivial solution** $\mathbf{x}=\mathbf{0}$. It may also have nontrivial solutions.

Consider

$$
\begin{cases}
x+y-z=0,\\
2x+2y-2z=0,\\
x-y+z=0.
\end{cases}
$$

In [ ]:
H0 = sp.Matrix([[1,1,-1,0],[2,2,-2,0],[1,-1,1,0]])
H1 = H0.copy(); H1[1,:]=H1[1,:]-2*H1[0,:]; H1[2,:]=H1[2,:]-H1[0,:]
H2 = H1.copy(); H2[2,:]=sp.Rational(-1,2)*H2[2,:]
H3 = H2.copy(); H3[0,:]=H3[0,:]-H3[2,:]
show_augmented(H0, "Initial homogeneous system")
show_augmented(H1, "Eliminate x")
show_augmented(H2, "Scale the nonzero row")
show_augmented(H3, "RREF")

The RREF equations are

$$
x=0,\qquad y-z=0.
$$

Let $z=t$. Then $y=t$, and

$$
(x,y,z)=(0,t,t)=t(0,1,1).
$$

The trivial solution occurs when $t=0$; every $t\ne0$ gives a nontrivial solution.

> **Common Mistakes in Section 1.2**
>
> - Changing only part of a row instead of applying the operation to the entire row.
> - Forgetting to apply a row operation to the augmented column.
> - Using a newly changed row inconsistently within the same step.
> - Stopping at REF and reading the solution without back-substitution.
> - Calling every zero row a contradiction. A contradiction occurs only when a row has the form
>
>   $$
>   [\,0\ \cdots\ 0\mid c\,], \qquad c
eq 0.
>   $$
>
> - Assuming that a homogeneous system has only the trivial solution.

## Section 1.2 terminology

| Term | Meaning |
|---|---|
| Matrix | Rectangular array of entries |
| $m\times n$ matrix | Matrix with $m$ rows and $n$ columns |
| Entry $a_{ij}$ | Entry in row $i$, column $j$ |
| Coefficient matrix | Matrix containing variable coefficients |
| Augmented matrix | Coefficient matrix with the constants column appended |
| Elementary row operation | Row replacement, row interchange, or nonzero row scaling |
| Row equivalent | Related by a sequence of elementary row operations |
| Leading entry | First nonzero entry in a nonzero row |
| Pivot position | Position of a leading entry in an echelon form |
| Row-echelon form | Stair-step form with zeros below pivots |
| Reduced row-echelon form | Echelon form with pivot 1s isolated in their columns |
| Gaussian elimination | Reduce to REF and back-substitute |
| Gauss–Jordan elimination | Reduce completely to RREF |
| Homogeneous system | A system $A\mathbf{x}=\mathbf0$ |
| Trivial solution | The zero solution of a homogeneous system |
| Nontrivial solution | Any nonzero solution of a homogeneous system |

## Section 1.2 Worksheet / In-Class Practice

1. State the dimensions and identify $a_{23}$ for

$$
A=\begin{bmatrix}
1&9&20\\
21&4&7
\end{bmatrix}.
$$

2. Write the augmented matrix for

$$
\begin{cases}
x-2y+z=9,\\
3x+y-4z=20.
\end{cases}
$$

3. Apply

$$
R_2\leftarrow R_2-3R_1
$$

to

$$
\left[
\begin{array}{cc|c}
1&2&9\\
3&7&30
\end{array}
\right].
$$

4. Determine whether each matrix is **REF**, **RREF**, or **neither**:

$$
A_1=
\begin{bmatrix}
1&2&-1&4\\
0&3&5&7\\
0&0&2&6
\end{bmatrix},
$$

$$
A_2=
\begin{bmatrix}
1&0&3\\
0&1&-2\\
0&0&0
\end{bmatrix},
$$

$$
A_3=
\begin{bmatrix}
1&2&0\\
0&0&1\\
0&3&0
\end{bmatrix}.
$$

5. Solve by Gaussian elimination:

$$
\begin{cases}
x+y+z=21,\\
2x+y+3z=41,\\
x+3y+2z=38.
\end{cases}
$$

6. Reduce to RREF:

$$
\left[
\begin{array}{ccc|c}
1&1&0&20\\
1&2&1&42\\
2&1&1&40
\end{array}
\right].
$$

7. Classify and solve:

$$
\begin{cases}
x+2y-z=9,\\
2x+4y-2z=18.
\end{cases}
$$

8. Find all solutions of the homogeneous system

$$
x+y+z=0,
\qquad
2x+2y+2z=0.
$$

Use the reveal buttons below one problem at a time.

## Section 1.2 Instructor Answer Reveals

Reveal one solution at a time during class.

In [ ]:
reveal_answer_key("1.2-1", "Reveal Problem 1")
reveal_answer_key("1.2-2", "Reveal Problem 2")
reveal_answer_key("1.2-3", "Reveal Problem 3")
reveal_answer_key("1.2-4", "Reveal Problem 4")
reveal_answer_key("1.2-5", "Reveal Problem 5")
reveal_answer_key("1.2-6", "Reveal Problem 6")
reveal_answer_key("1.2-7", "Reveal Problem 7")
reveal_answer_key("1.2-8", "Reveal Problem 8")

# Section 1.3: Applications of Linear Systems

## Learning objectives

Students should be able to:

- translate an applied situation into variables, equations, and a matrix model;
- fit lines and quadratic polynomials through data points;
- solve and interpret network-flow systems;
- apply conservation laws and physical restrictions;
- formulate simple circuit systems using Kirchhoff's laws;
- use Python to visualize and explore how parameter changes affect a model.

## 1. A modeling workflow

A reliable applied-linear-algebra workflow is:

1. Define variables with units.
2. Translate each constraint into an equation.
3. Assemble $A\mathbf{x}=\mathbf b$.
4. Solve or parameterize the system.
5. Check mathematical and physical constraints.
6. Interpret the result in the original context.

> **Modeling Warning**
>
> A mathematically correct solution can still be physically impossible. Negative traffic flow, negative product quantities, or an unexpected current direction require interpretation rather than automatic rejection.

## 2. Curve fitting: a line through two points

Suppose an original teaching dataset contains the points

$$
(1,21),\qquad(3,9).
$$

We seek a line

$$
y=mx+b.
$$

Substituting the two points gives

$$
\begin{cases}
m+b=21,\\
3m+b=9.
\end{cases}
$$

Subtract the first equation from the second:

$$
2m=-12\quad\Longrightarrow\quad m=-6.
$$

Then

$$
-6+b=21\quad\Longrightarrow\quad b=27.
$$

Thus

$$
y=-6x+27.
$$

In [ ]:
m,b = sp.symbols('m b')
line_solution = sp.solve([sp.Eq(m+b,21), sp.Eq(3*m+b,9)],[m,b], dict=True)[0]
line_solution

In [ ]:
xplot = np.linspace(0,4,200)
yplot = -6*xplot+27
plt.figure(figsize=(7,5))
plt.plot(xplot,yplot,label=r'$y=-6x+27$')
plt.scatter([1,3],[21,9],s=80,label='given data')
plt.xlabel('x'); plt.ylabel('y'); plt.title('Exact line interpolation')
plt.grid(alpha=0.3); plt.legend(); plt.show()

## 3. Quadratic interpolation through three points

Fit

$$
p(x)=ax^2+bx+c
$$

through the original, jersey-number-inspired points

$$
(0,9),\qquad(1,20),\qquad(2,21).
$$

Substitute each point:

- At $x=0$: $c=9$.
- At $x=1$: $a+b+c=20$.
- At $x=2$: $4a+2b+c=21$.

The coefficient system is

$$
\begin{cases}
c=9,\\
a+b+c=20,\\
4a+2b+c=21.
\end{cases}
$$

In matrix form,

$$
\left[
\begin{array}{ccc|c}
0&0&1&9\\
1&1&1&20\\
4&2&1&21
\end{array}
\right].
$$

In [ ]:
Q0 = sp.Matrix([[0,0,1,9],[1,1,1,20],[4,2,1,21]])
show_augmented(Q0,"Quadratic interpolation system")

Use $R_2\leftarrow R_2-R_1$ and $R_3\leftarrow R_3-R_1$:

$$
\left[
\begin{array}{ccc|c}
0&0&1&9\\
1&1&0&11\\
4&2&0&12
\end{array}
\right].
$$

Now eliminate $a$ from Row 3 using

$$
R_3\leftarrow R_3-4R_2:
$$

$$
[4,2,0\mid12]-4[1,1,0\mid11]=[0,-2,0\mid-32].
$$

Thus

$$
-2b=-32\quad\Longrightarrow\quad b=16.
$$

From $a+b=11$,

$$
a=11-16=-5.
$$

Since $c=9$,

$$
\boxed{p(x)=-5x^2+16x+9}.
$$

In [ ]:
Q1 = Q0.copy(); Q1[1,:]=Q1[1,:]-Q1[0,:]; Q1[2,:]=Q1[2,:]-Q1[0,:]
Q2 = Q1.copy(); Q2[2,:]=Q2[2,:]-4*Q2[1,:]
show_augmented(Q1,"After removing c from Rows 2 and 3")
show_augmented(Q2,"After eliminating a from Row 3")

x = sp.symbols('x')
p = -5*x**2 + 16*x + 9
print("Polynomial:", p)
print("Check values:", [p.subs(x,k) for k in [0,1,2]])

In [ ]:
xplot=np.linspace(-0.4,2.6,300)
yplot=-5*xplot**2+16*xplot+9
plt.figure(figsize=(7,5))
plt.plot(xplot,yplot,label=r'$p(x)=-5x^2+16x+9$')
plt.scatter([0,1,2],[9,20,21],s=80,label='interpolation points')
plt.xlabel('x'); plt.ylabel('p(x)'); plt.title('Quadratic interpolation')
plt.grid(alpha=0.3); plt.legend(); plt.show()

### Interactive quadratic fitting

Change the three $y$-values while keeping $x=0,1,2$. Python rebuilds the linear system and displays the unique quadratic.

In [ ]:
if WIDGETS_AVAILABLE:
    @interact(
        y0=IntSlider(value=9,min=-10,max=40,description='y(0)'),
        y1=IntSlider(value=20,min=-10,max=40,description='y(1)'),
        y2=IntSlider(value=21,min=-10,max=40,description='y(2)'))
    def interactive_quadratic(y0,y1,y2):
        A=sp.Matrix([[0,0,1],[1,1,1],[4,2,1]])
        coeff=A.LUsolve(sp.Matrix([y0,y1,y2]))
        aa,bb,cc=map(float,coeff)
        xx=np.linspace(-0.5,2.5,300)
        yy=aa*xx**2+bb*xx+cc
        plt.figure(figsize=(7,5)); plt.plot(xx,yy)
        plt.scatter([0,1,2],[y0,y1,y2],s=80)
        plt.grid(alpha=0.3); plt.title(f"p(x)={aa:.2f}x²+{bb:.2f}x+{cc:.2f}")
        plt.show()
        display(coeff)
else:
    print("Open in Colab for interactive quadratic fitting.")

## 4. Network flow and conservation

At every intersection, conservation requires

$$
\text{total inflow}=\text{total outflow}.
$$

Consider the original traffic network below. The values are vehicles per hour.

In [ ]:
# Draw a simple directed network using Matplotlib.
fig, ax = plt.subplots(figsize=(9,5))
nodes = {'A':(0,1),'B':(2,2),'C':(2,0),'D':(4,1)}
for name,(xx,yy) in nodes.items():
    ax.scatter([xx],[yy],s=900)
    ax.text(xx,yy,name,ha='center',va='center',fontsize=13)

def arrow(start,end,label,offset=(0,0)):
    x1,y1=nodes[start]; x2,y2=nodes[end]
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',lw=2))
    ax.text((x1+x2)/2+offset[0],(y1+y2)/2+offset[1],label,fontsize=12)

arrow('A','B','$x_1$',(0,0.15)); arrow('A','C','$x_2$',(0,-0.2))
arrow('B','D','$x_3$',(0,0.15)); arrow('C','D','$x_4$',(0,-0.2))
ax.annotate('',xy=(-0.05,1),xytext=(-1,1),arrowprops=dict(arrowstyle='->',lw=2)); ax.text(-0.75,1.18,'67')
ax.annotate('',xy=(2,2.05),xytext=(2,2.9),arrowprops=dict(arrowstyle='->',lw=2)); ax.text(2.1,2.55,'20')
ax.annotate('',xy=(4.9,1),xytext=(4.05,1),arrowprops=dict(arrowstyle='->',lw=2)); ax.text(4.45,1.18,'66')
ax.annotate('',xy=(2,-0.05),xytext=(2,-0.9),arrowprops=dict(arrowstyle='->',lw=2)); ax.text(2.1,-0.55,'21')
ax.set_xlim(-1.2,5.2); ax.set_ylim(-1.1,3.1); ax.axis('off'); ax.set_title('Traffic-flow network')
plt.show()

The node equations are chosen as follows:

- Node $A$: $x_1+x_2=67$.
- Node $B$: $x_1+20=x_3$.
- Node $C$: $x_2=x_4+21$.
- Node $D$: $x_3+x_4=66$. This equation is dependent on the first three balances.

Use the first three equations to expose the free variable. Let

$$
x_1=t.
$$

Then

$$
x_2=67-t,
$$

and

$$
x_3=t+20.
$$

Substitute into the Node $C$ equation:

$$
67-t=x_4+21.
$$

Therefore

$$
x_4=46-t.
$$

The Node $D$ balance is then automatic:

$$
(t+20)+(46-t)=66.
$$

Thus

$$
(x_1,x_2,x_3,x_4)=(t,67-t,t+20,46-t).
$$

For all flows to be nonnegative,

$$
0\le t\le46.
$$

In [ ]:
t = sp.symbols('t', real=True)
flow_vector = sp.Matrix([t,67-t,t+20,46-t])
flow_vector

In [ ]:
if WIDGETS_AVAILABLE:
    @interact(t_value=IntSlider(value=21,min=0,max=46,description='x1=t'))
    def traffic_parameter(t_value):
        flows=[t_value,67-t_value,t_value+20,46-t_value]
        labels=['x1','x2','x3','x4']
        plt.figure(figsize=(7,4)); plt.bar(labels,flows)
        plt.ylim(0,90); plt.ylabel('vehicles per hour')
        plt.title(f"Feasible flow for t={t_value}")
        for i,val in enumerate(flows): plt.text(i,val+2,str(val),ha='center')
        plt.show()
else:
    print("Enable ipywidgets to use the traffic-flow parameter slider.")

> **Interpretation**
>
> The system does not determine a unique internal traffic pattern. It determines a one-parameter family. Operational restrictions, lane capacities, or measured flows would be needed to choose a specific value of $t$.

## 5. Electrical networks and Kirchhoff's laws

For a simple circuit model:

- **Kirchhoff's Current Law:** current entering a junction equals current leaving it.
- **Kirchhoff's Voltage Law:** the signed voltage changes around a closed loop sum to zero.
- **Ohm's law:** $V=RI$.

In [ ]:
# Junction schematic matching I1 + I3 = I2.
fig, ax = plt.subplots(figsize=(9,5))
J=(2.4,1.2)
ax.scatter([J[0]],[J[1]],s=500)
ax.text(J[0],J[1],'J',ha='center',va='center',fontsize=13)
ax.annotate('',xy=J,xytext=(0.2,2.2),arrowprops=dict(arrowstyle='->',lw=2))
ax.annotate('',xy=J,xytext=(0.2,0.2),arrowprops=dict(arrowstyle='->',lw=2))
ax.annotate('',xy=(4.8,1.2),xytext=J,arrowprops=dict(arrowstyle='->',lw=2))
ax.text(1.1,1.95,'$I_1$',fontsize=14)
ax.text(1.1,0.35,'$I_3$',fontsize=14)
ax.text(3.55,1.38,'$I_2$',fontsize=14)
ax.set_xlim(-0.2,5.2);ax.set_ylim(-0.2,2.5);ax.axis('off')
ax.set_title('Junction-current schematic: $I_1+I_3=I_2$')
plt.show()

Suppose the model produces

$$
\begin{cases}
I_1-I_2+I_3=0,\\
2I_1+I_2=38,\\
I_2+3I_3=53.
\end{cases}
$$

The first equation is a junction balance. The last two are loop equations.

In [ ]:
C0 = sp.Matrix([[1,-1,1,0],[2,1,0,38],[0,1,3,53]])
show_augmented(C0,"Initial circuit system")

Use Row 1 as the first pivot row. Eliminate $I_1$ from Row 2:

$$
R_2\leftarrow R_2-2R_1.
$$

$$
[2,1,0\mid38]-2[1,-1,1\mid0]=[0,3,-2\mid38].
$$

In [ ]:
C1=C0.copy(); C1[1,:]=C1[1,:]-2*C1[0,:]
show_augmented(C1,r"After R2 <- R2 - 2R1")

Now eliminate $I_2$ from Row 3. A fraction-free choice is

$$
R_3\leftarrow 3R_3-R_2.
$$

First,

$$
3R_3=[0,3,9\mid159].
$$

Then

$$
[0,3,9\mid159]-[0,3,-2\mid38]=[0,0,11\mid121].
$$

In [ ]:
C2=C1.copy(); C2[2,:]=3*C2[2,:]-C2[1,:]
show_augmented(C2,r"After R3 <- 3R3 - R2")

Back-substitute:

$$
11I_3=121\quad\Longrightarrow\quad I_3=11.
$$

From Row 2,

$$
3I_2-2I_3=38,
$$

so

$$
3I_2-22=38\quad\Longrightarrow\quad3I_2=60\quad\Longrightarrow\quad I_2=20.
$$

From Row 1,

$$
I_1-I_2+I_3=0,
$$

so

$$
I_1-20+11=0\quad\Longrightarrow\quad I_1=9.
$$

Therefore,

$$
(I_1,I_2,I_3)=(9,20,11).
$$

In [ ]:
A_circuit=sp.Matrix([[1,-1,1],[2,1,0],[0,1,3]])
b_circuit=sp.Matrix([0,38,53])
A_circuit.LUsolve(b_circuit)

### Interactive voltage sensitivity

Vary the two loop-voltage constants and observe how all three currents change.

In [ ]:
if WIDGETS_AVAILABLE:
    @interact(V1=IntSlider(value=38,min=10,max=70,description='V1'),
              V2=IntSlider(value=53,min=10,max=90,description='V2'))
    def circuit_sensitivity(V1,V2):
        A=np.array([[1,-1,1],[2,1,0],[0,1,3]],dtype=float)
        currents=np.linalg.solve(A,np.array([0,V1,V2],dtype=float))
        labels=['I1','I2','I3']
        plt.figure(figsize=(7,4)); plt.bar(labels,currents)
        plt.axhline(0,linewidth=0.8); plt.ylabel('current (model units)')
        plt.title(f"Currents for V1={V1}, V2={V2}")
        for i,val in enumerate(currents): plt.text(i,val+0.5 if val>=0 else val-1.5,f"{val:.2f}",ha='center')
        plt.show()
else:
    print("Enable ipywidgets to use circuit sensitivity sliders.")

## 6. Optional application: color-sensor calibration

A camera sensor combines unknown red, green, and blue response weights $(r,g,b)$. Three calibration cards produce

$$
\begin{cases}
r+g+b=30,\\
2r+g+3b=41,\\
r+4g+2b=91.
\end{cases}
$$

This is a linear calibration model. Python can solve it exactly, but students should still inspect the coefficient matrix and verify the result.

In [ ]:
A_rgb=sp.Matrix([[1,1,1],[2,1,3],[1,4,2]])
b_rgb=sp.Matrix([30,41,91])
rgb_solution=A_rgb.LUsolve(b_rgb)
rgb_solution, A_rgb*rgb_solution

### Classroom Discussion

Ask students: **What assumptions make each of these application models linear?**

Consider curve fitting, network flow, and electrical circuits. Reveal the suggested discussion points after students respond.

In [ ]:
reveal_answer_key("application-discussion", "Reveal discussion points")

> **Common Mistakes in Section 1.3**
>
> - Failing to define variables and units.
> - Using data coordinates in the wrong polynomial order.
> - Writing inflow minus outflow inconsistently from node to node.
> - Ignoring nonnegativity or capacity restrictions.
> - Treating a free parameter as a failure rather than meaningful flexibility.
> - Reporting Python output without checking it in the original equations.

## Section 1.3 terminology

| Term | Meaning |
|---|---|
| Mathematical model | Mathematical representation of a real or designed situation |
| Interpolation | Constructing a function that passes exactly through specified data points |
| Polynomial coefficients | Unknown constants $a,b,c,\ldots$ in a polynomial model |
| Vandermonde-type system | Linear system created by substituting data points into a polynomial |
| Network | Collection of nodes connected by directed or undirected links |
| Node | Junction at which flow is conserved |
| Flow conservation | Total inflow equals total outflow |
| Free-flow parameter | Parameter representing an undetermined internal flow |
| Feasible solution | Mathematically valid solution satisfying physical restrictions |
| Kirchhoff's Current Law | Current entering a junction equals current leaving it |
| Kirchhoff's Voltage Law | Signed voltage changes around a closed loop sum to zero |
| Ohm's law | Relationship $V=RI$ |
| Sensitivity | How model outputs change when inputs or parameters change |

## Section 1.3 Worksheet / In-Class Practice

1. Find the line through $(2,21)$ and $(5,9)$.

2. Find the quadratic

$$
p(x)=ax^2+bx+c
$$

passing through $(0,1)$, $(1,9)$, and $(2,21)$.

3. A node receives $50$ units from outside and sends flows $x_1,x_2$ outward. Write the conservation equation.

4. Solve and interpret the flow family:

$$
x_1+x_2=50,
\qquad
x_1+9=x_3,
\qquad
x_2+x_3=59.
$$

5. For the family in Problem 4, determine the feasible parameter range if all flows must be nonnegative.

6. Solve the circuit model:

$$
I_1-I_2+I_3=0,
\qquad
I_1+I_2=21,
\qquad
I_2+2I_3=20.
$$

7. A three-channel calibration model satisfies

$$
r+g+b=21,
\qquad
2r+g+b=30,
\qquad
r+2g+3b=44.
$$

Find $(r,g,b)$ and verify the result.

Use the reveal buttons below one problem at a time.

## Section 1.3 Instructor Answer Reveals

Reveal one solution at a time during class.

In [ ]:
reveal_answer_key("1.3-1", "Reveal Problem 1")
reveal_answer_key("1.3-2", "Reveal Problem 2")
reveal_answer_key("1.3-3", "Reveal Problem 3")
reveal_answer_key("1.3-4", "Reveal Problem 4")
reveal_answer_key("1.3-5", "Reveal Problem 5")
reveal_answer_key("1.3-6", "Reveal Problem 6")
reveal_answer_key("1.3-7", "Reveal Problem 7")

# Chapter 1 wrap-up

## Mathematical storyline

- Linear equations encode constraints.
- A system asks for values satisfying all constraints simultaneously.
- Geometry explains the three solution possibilities.
- Matrices organize coefficients and constants.
- Row operations preserve the solution set.
- Gaussian elimination reaches REF and uses back-substitution.
- Gauss–Jordan elimination reaches RREF.
- Parameters describe solution families.
- Applications require interpretation and feasibility checks.

## Python reference used in this notebook

| Task | Recommended command |
|---|---|
| Create a NumPy matrix | `np.array([[...], [...]])` |
| Create an exact SymPy matrix | `sp.Matrix([[...], [...]])` |
| Solve a square numerical system | `np.linalg.solve(A, b)` |
| Solve an exact symbolic system | `A.LUsolve(b)` or `sp.linsolve(...)` |
| Compute RREF | `M.rref()` |
| Compute rank | `M.rank()` |
| Join $A$ and $b$ | `A.row_join(b)` |
| Verify a solution | compute `A*x` and compare with `b` |
| Plot lines/curves | `plt.plot(...)` |
| Plot points | `plt.scatter(...)` |
| Add interactivity in Colab | `ipywidgets.interact` |

> **Closing Note for the Instructor**
>
> This notebook intentionally includes more material than one class meeting requires. The examples are modular. Skip optional applications or interactive cells when time is short, but do not skip the intermediate matrix calculations when introducing an algorithm.
>
> Students should repeatedly see the bridge:
>
> $$
> \text{equation system}\;\longrightarrow\;\text{augmented matrix}\;\longrightarrow\;\text{row operations}\;\longrightarrow\;\text{solution}\;\longrightarrow\;\text{interpretation}.
> $$